# Multi-Channel sEMG Fatigue Analysis: Temporal Feature Trends and Subject-Independent Classification of Early vs Late Recording Stages

This notebook analyses multichannel surface EMG (sEMG) recorded during a fatigue protocol (15 subjects, 8 channels, 200 Hz, 120 s retained per subject). It examines how EMG-derived features change over time and how well those features discriminate early from late segments of the recording in subjects not used for model fitting. The dataset and the reference publications are described in the repository README.

**Scope.** The analysis characterizes feature changes over recording time. Early and late labels are defined by time within the recording and serve as proxies for fatigue-related states; the notebook does not perform clinical fatigue detection.

## Contents
1. Data Loading and Quality Control
2. Epoch-Based EMG Feature Extraction
3. Temporal Trend Analysis
4. Spectral Feature Analysis
5. Fatigue-Zone Analysis
6. Temporal Trajectory Modeling
7. Exploratory Fatigue-Response Ranking
8. Feature Extraction for Window Classification
9. Window Dataset and Subject-Wise Evaluation Framework
10. Binary Temporal-Window Classification
11. Three-Class Temporal-Window Classification
12. Sensitivity Analysis
13. Channel Contribution Analysis
14. Model Feature Importance
15. Summary of Classification Performance
16. Limitations and Interpretation
17. Export of Key Results

## Key definitions used in the classification sections
- **Epoch:** 1 s of signal (200 samples) per channel. All features are computed per epoch after removing the epoch mean.
- **Window:** mean of the epoch features over 5 consecutive epochs, moved in 1 s steps.
- **Binary labels (early vs late):** windows lying entirely within the first 25% of the recording are *early*; windows starting in the last 25% are *late*; intermediate windows are excluded.
- **Three-class labels (early / middle / late):** windows lying entirely within the first, second or third third of the recording.
- **Baseline normalization:** each feature is expressed relative to the same subject's mean over the first 10 s of the recording (relative change; log-scale features by subtraction). Windows that start inside this 10 s baseline are excluded from every classification experiment. Normalization therefore requires an initial baseline segment for every subject.
- **Leave-One-Subject-Out (LOSO):** each subject is held out in turn; all learned preprocessing (standardization) and model fitting use the remaining 14 subjects only. Subject-level summaries (n = 15) are the unit of statistical inference; windows overlap and are not independent observations.

## Running the notebook
1. **Dependencies:** `numpy`, `scipy`, `pandas`, `matplotlib`, `scikit-learn` and `jupyter` (see `requirements.txt`). `h5py` is needed only if the `.mat` file is in MATLAB v7.3 (HDF5) format.
2. **Dataset:** the recordings are not distributed with this notebook (see the repository README for the source and terms). The notebook looks for `All_15sub_Data.mat` in this order: the environment variable `MAT_PATH`; `<PROJECT_DIR>/data/All_15sub_Data.mat`; `<PROJECT_DIR>/All_15sub_Data.mat`. `PROJECT_DIR` defaults to the current working directory and can be set with an environment variable of the same name.
3. **Run all cells in order.** Only the key outputs are exported for use outside the notebook (Section 17): 8 selected figures in `<PROJECT_DIR>/results/figures`, key tables and a headline-results summary in `<PROJECT_DIR>/results/tables`. Complete intermediate tables are kept in `<PROJECT_DIR>/results/supplementary`. To keep all results in Google Drive (or any synced folder), set `PROJECT_DIR` to that folder. The permutation test in Section 10.4 is the slowest step; reduce `N_PERM` in Section 9.1 for a quick test run.

## 1. Data Loading and Quality Control
### 1.1 Setup and Configuration

In [ ]:
import itertools
import os
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from scipy import io, signal, stats
from scipy.optimize import curve_fit
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score,
                             confusion_matrix, f1_score, roc_auc_score)
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# Paths (configurable through the environment variables PROJECT_DIR and MAT_PATH)
PROJECT_DIR = os.environ.get("PROJECT_DIR", ".")
_default_mats = [os.path.join(PROJECT_DIR, "data", "All_15sub_Data.mat"),
                 os.path.join(PROJECT_DIR, "All_15sub_Data.mat")]
MAT_PATH = os.environ.get("MAT_PATH") or next((p for p in _default_mats if os.path.exists(p)), _default_mats[0])
if not os.path.exists(MAT_PATH):
    raise FileNotFoundError(f"Dataset not found: {MAT_PATH}. Place All_15sub_Data.mat in '{_default_mats[0]}' "
                            "or set the MAT_PATH environment variable (see 'Running the notebook').")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
FIG_DIR = os.path.join(RESULTS_DIR, "figures")         # selected key figures (KEY_FIGURES below)
TABLE_DIR = os.path.join(RESULTS_DIR, "tables")        # key tables and headline summary (Section 17)
OUT_DIR = os.path.join(RESULTS_DIR, "supplementary")   # complete intermediate tables and epoch-feature archive
for _d in (FIG_DIR, TABLE_DIR, OUT_DIR):
    os.makedirs(_d, exist_ok=True)

# Acquisition and analysis constants
FS = 200               # sampling frequency (Hz)
N_CH = 8               # EMG channels per subject
PSD_METHOD = "stft"    # "stft": Hann-windowed FFT per 1 s epoch | "welch": Welch estimate within each epoch
MDF_FMIN = 0.0         # lower integration limit of the median frequency (Hz)
ALPHA = 0.05           # significance level for descriptive tests
SEED = 42
np.random.seed(SEED)

plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 100})

# Figures exported to results/figures: the ones that carry the story of the analysis.
# All other figures are displayed in the notebook only (set SAVE_ALL_FIGURES = True to also save them to results/figures_all).
KEY_FIGURES = [
    "mean_slopes_ci.png",                 # 3.3  ΔMDF / ΔRMS per subject and channel with 95% CIs
    "linear_fit_r2_histograms.png",       # 3.1  how well a straight line describes RMS and MDF trajectories
    "spectral_index_comparison.png",      # 4    RMS vs MDF and alternative spectral indices
    "fatigue_zone_analysis.png",          # 5    fatigue zones per series, subject and channel
    "trajectory_example_fits.png",        # 6    linear vs piecewise vs exponential model (example series)
    "binary_accuracy_by_feature_set.png", # 10   LOSO early-vs-late accuracy, normalized vs raw features
    "three_class_confusion_matrix.png",   # 11   three-class temporal-window confusion matrix
    "channel_contribution_analysis.png",  # 13   single-channel, drop-one and top-k channel analysis
]
SAVE_ALL_FIGURES = False
FIG_EXTRA_DIR = os.path.join(RESULTS_DIR, "figures_all")

def save_show(fname, **kwargs):
    """Apply tight layout, save the figure if it is a key figure (or SAVE_ALL_FIGURES is True), then display it."""
    plt.tight_layout()
    if fname in KEY_FIGURES:
        plt.savefig(os.path.join(FIG_DIR, fname), dpi=150, **kwargs)
    elif SAVE_ALL_FIGURES:
        os.makedirs(FIG_EXTRA_DIR, exist_ok=True)
        plt.savefig(os.path.join(FIG_EXTRA_DIR, fname), dpi=150, **kwargs)
    plt.show()

print("Dataset:", MAT_PATH)
print("Figures ->", FIG_DIR, "| key tables ->", TABLE_DIR, "| supplementary ->", OUT_DIR)
print("numpy", np.__version__, "| scipy", __import__("scipy").__version__, "| pandas", pd.__version__,
      "| matplotlib", __import__("matplotlib").__version__, "| scikit-learn", __import__("sklearn").__version__)

### 1.2 Loading the Recordings
The loader accepts a cell array, a struct, a 3-D array, or one variable per subject, and returns one `(samples, 8)` array per subject. All recordings are truncated to the shortest recording so that every subject has the same number of 1 s epochs.

In [ ]:
def natural_key(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", str(s))]

def to_samples_by_ch(a):
    """Return an array of shape (n_samples, N_CH), or None if `a` is not an EMG matrix."""
    a = np.asarray(a)
    if a.dtype == object or not np.issubdtype(a.dtype, np.number):
        return None
    a = a.astype(float).squeeze()
    if a.ndim != 2:
        return None
    if a.shape[1] == N_CH and a.shape[0] > N_CH:
        return a
    if a.shape[0] == N_CH and a.shape[1] > N_CH:
        return a.T
    return None

def collect(obj, name, out):
    """Recursively collect candidate subject arrays from a loaded .mat object."""
    if isinstance(obj, io.matlab.mat_struct):
        for f in obj._fieldnames:
            collect(getattr(obj, f), f"{name}.{f}", out)
    elif isinstance(obj, np.ndarray) and obj.dtype == object:
        for i, o in enumerate(obj.ravel()):
            collect(o, f"{name}[{i}]", out)
    elif isinstance(obj, np.ndarray) and np.issubdtype(obj.dtype, np.number) and obj.ndim == 3 and N_CH in obj.shape:
        t_ax = int(np.argmax(obj.shape))                                   # longest axis = time
        c_ax = [i for i, n in enumerate(obj.shape) if n == N_CH and i != t_ax][0]
        s_ax = [i for i in range(3) if i not in (t_ax, c_ax)][0]
        arr = np.moveaxis(obj, [s_ax, t_ax, c_ax], [0, 1, 2]).astype(float)
        for i in range(arr.shape[0]):
            out.append((f"{name}[{i}]", arr[i]))
    else:
        a = to_samples_by_ch(obj) if isinstance(obj, np.ndarray) else None
        if a is not None:
            out.append((name, a))

def load_subjects(path):
    found = []
    try:
        m = io.loadmat(path, squeeze_me=True, struct_as_record=False)
        keys = sorted([k for k in m if not k.startswith("__")], key=natural_key)
        print("Variables in .mat:", [(k, getattr(m[k], "shape", type(m[k]).__name__)) for k in keys])
        for k in keys:
            collect(m[k], k, found)
    except NotImplementedError:  # MATLAB v7.3 files (HDF5)
        import h5py
        print("MATLAB v7.3 file: using h5py")
        with h5py.File(path, "r") as f:
            def visit(name, ds):
                if isinstance(ds, h5py.Dataset):
                    a = to_samples_by_ch(np.array(ds))
                    if a is not None:
                        found.append((name, a))
            f.visititems(visit)
            found.sort(key=lambda x: natural_key(x[0]))
    return found

found = load_subjects(MAT_PATH)
print(f"\nFound {len(found)} subject arrays")
for i, (n, a) in enumerate(found, 1):
    print(f"  Sub{i:<2d} <- {n:30s} shape={a.shape}")
assert len(found) > 0, "No EMG arrays found: check MAT_PATH and the variable structure printed above."
if len(found) != 15:
    print("WARNING: expected 15 subjects. Check the list above; subject numbering follows this order.")

# Subject numbering (Sub1, Sub2, ...) follows the order of the arrays found in the file
data = [a for _, a in found]                       # list of (n_samples, N_CH) arrays
n_min = min(a.shape[0] for a in data)
E = n_min // FS                                    # number of 1 s epochs per subject (120 expected)
X = np.stack([a[:E * FS] for a in data])           # (S, E * FS, N_CH)
S = X.shape[0]
print(f"\nX shape = {X.shape}: {S} subjects, {E} epochs of 1 s, {N_CH} channels")

### 1.3 Signal Quality Summary
Per-subject checks of recording length, amplitude range, clipping, mean offset, missing values and flat channels, followed by the average power spectrum. The data are band-limited and notch-filtered at 50 Hz, which lies inside the median-frequency range; the spectrum is inspected for the resulting dip.

In [ ]:
rows = []
for s in range(S):
    d = data[s]
    rows.append(dict(subject=f"Sub{s+1}", samples=d.shape[0], duration_s=round(d.shape[0] / FS, 1),
                     min=d.min(), max=d.max(),
                     clipped_pct=round(100 * np.mean((d <= -128) | (d >= 127)), 3),   # 8-bit range limits
                     mean_abs_offset=round(np.abs(d.mean(0)).mean(), 3),
                     n_nan=int(np.isnan(d).sum()),
                     n_flat_channels=int((d.std(0) < 1e-6).sum())))
qc = pd.DataFrame(rows)
display(qc)
qc.to_csv(os.path.join(OUT_DIR, "qc_report.csv"), index=False)
assert qc.n_nan.sum() == 0, "Recordings contain NaN values."

f_, P_ = signal.welch(X - X.mean(1, keepdims=True), fs=FS, nperseg=FS, axis=1)
plt.figure(figsize=(7, 3.5))
plt.semilogy(f_, P_.mean(axis=(0, 2)))
plt.axvline(50, ls="--", c="r", label="50 Hz")
plt.xlabel("Frequency (Hz)"); plt.ylabel("Power spectral density (a.u.)")
plt.title("Mean power spectral density across subjects and channels")
plt.legend()
save_show("qc_mean_psd.png")

## 2. Epoch-Based EMG Feature Extraction
For each subject, channel and 1 s epoch the mean is removed and two features are computed:
- **RMS** of the demeaned raw signal.
- **Median frequency (MDF):** the frequency that splits the epoch power spectrum into two halves of equal power (linearly interpolated between spectral bins). The spectrum is a Hann-windowed FFT power spectrum (`PSD_METHOD = "stft"`) integrated from `MDF_FMIN` to the Nyquist frequency.

In [ ]:
def psd_epochs(segs):
    """segs: (E, FS, C) -> frequency axis (F,), power spectrum (E, F, C)."""
    if PSD_METHOD == "stft":
        w = signal.windows.hann(segs.shape[1], sym=False)[None, :, None]
        P = np.abs(np.fft.rfft(segs * w, axis=1)) ** 2
        f = np.fft.rfftfreq(segs.shape[1], 1 / FS)
    else:
        f, P = signal.welch(segs, fs=FS, nperseg=100, noverlap=50, axis=1)
    return f, P

def median_freq(f, P, fmin=0.0):
    """Frequency splitting the power spectrum into two equal-power halves, linearly interpolated."""
    m = f >= fmin
    f, P = f[m], P[:, m, :]
    c = np.cumsum(P, axis=1)
    half = c[:, -1:, :] / 2
    idx = np.argmax(c >= half, axis=1)                         # (E, C)
    idx0 = np.clip(idx - 1, 0, None)
    c1 = np.take_along_axis(c, idx[:, None, :], 1)[:, 0]
    c0 = np.take_along_axis(c, idx0[:, None, :], 1)[:, 0]
    frac = np.clip(np.where(c1 - c0 > 0, (half[:, 0] - c0) / (c1 - c0), 0), 0, 1)
    return f[idx0] + frac * (f[idx] - f[idx0])

rms_ep = np.zeros((S, E, N_CH))
mdf_ep = np.zeros((S, E, N_CH))
for s in range(S):
    segs = X[s].reshape(E, FS, N_CH)
    segs = segs - segs.mean(axis=1, keepdims=True)             # remove the DC offset of each epoch
    rms_ep[s] = np.sqrt((segs ** 2).mean(axis=1))
    f_ax, P = psd_epochs(segs)
    mdf_ep[s] = median_freq(f_ax, P, MDF_FMIN)

assert np.isfinite(rms_ep).all() and np.isfinite(mdf_ep).all(), "Non-finite RMS/MDF values."
print("rms_ep:", rms_ep.shape, "| mdf_ep:", mdf_ep.shape)
print(f"MDF range: {mdf_ep.min():.1f}-{mdf_ep.max():.1f} Hz | mean MDF, first 10 s: {mdf_ep[:, :10].mean():.1f} Hz; last 10 s: {mdf_ep[:, -10:].mean():.1f} Hz")
np.savez(os.path.join(OUT_DIR, "epoch_features.npz"), rms_ep=rms_ep, mdf_ep=mdf_ep)

sub_lbl = [f"Sub{i+1}" for i in range(S)]
ch_lbl = [f"Ch{i+1}" for i in range(N_CH)]

## 3. Temporal Trend Analysis
### 3.1 Linear Trends of RMS and MDF
A straight line is fitted to every subject × channel series (epoch index in seconds). The slope (ΔRMS in RMS units per second, ΔMDF in Hz per second) quantifies the average rate of change; R² describes how well a straight line represents the trajectory.

In [ ]:
def fit_all(F):
    """F: (S, E, C) -> slope, R², intercept and residual SD, each of shape (S, C)."""
    t = np.arange(F.shape[1])
    slope = np.zeros((F.shape[0], F.shape[2])); r2 = slope.copy(); icpt = slope.copy(); rsd = slope.copy()
    for s in range(F.shape[0]):
        for c in range(F.shape[2]):
            y = F[s, :, c]
            r = stats.linregress(t, y)
            slope[s, c], r2[s, c], icpt[s, c] = r.slope, r.rvalue ** 2, r.intercept
            rsd[s, c] = np.std(y - (r.intercept + r.slope * t), ddof=2)
    return slope, r2, icpt, rsd

d_rms, r2_rms, _, _ = fit_all(rms_ep)
d_mdf, r2_mdf, ic_mdf, sd_mdf = fit_all(mdf_ep)

for name, M in [("delta_rms", d_rms), ("delta_mdf", d_mdf), ("r2_rms", r2_rms), ("r2_mdf", r2_mdf)]:
    pd.DataFrame(M, index=sub_lbl, columns=ch_lbl).to_csv(os.path.join(OUT_DIR, f"{name}.csv"))

print(f"Median R² of the linear fit: RMS {np.median(r2_rms):.3f} | MDF {np.median(r2_mdf):.3f}")
print(f"Share of fits with R² < 0.1: RMS {np.mean(r2_rms < 0.1):.1%} | MDF {np.mean(r2_mdf < 0.1):.1%}")
print(f"Subject-channel series with ΔRMS > 0: {np.mean(d_rms > 0):.1%} | with ΔMDF < 0: {np.mean(d_mdf < 0):.1%}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for a, R, n in zip(ax, [r2_rms, r2_mdf], ["RMS", "MDF"]):
    a.hist(R.ravel(), bins=20, color="#2a7ab0")
    a.set_title(f"Linear-fit R² across subject-channel series: {n}"); a.set_xlabel("R²"); a.set_ylabel("Number of series")
save_show("linear_fit_r2_histograms.png")

### 3.2 Statistical Analysis of Temporal Trends
One-sample t-tests and Wilcoxon signed-rank tests of the slopes against zero, with Holm and Benjamini–Hochberg correction and 95% confidence intervals. *Channel-level* tests use the 15 subjects as observations for each channel; *subject-level* tests use the 8 channels as observations for each subject (channels of one subject are correlated, so these are descriptive). The overall effect uses subject means.

In [ ]:
def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); adj = np.empty_like(p); run = 0.0; m = len(p)
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

def bh(p):
    p = np.asarray(p, float); m = len(p); order = np.argsort(p)
    adj = np.empty_like(p); prev = 1.0
    for rank in range(m - 1, -1, -1):
        i = order[rank]; prev = min(prev, p[i] * m / (rank + 1)); adj[i] = prev
    return adj

def slope_table(M, axis, labels):
    """M: (S, C). axis=0: one row per channel (test across subjects); axis=1: one row per subject (test across channels)."""
    rows = []
    for i, lab in enumerate(labels):
        v = M[:, i] if axis == 0 else M[i, :]
        mean = v.mean(); sem = stats.sem(v)
        lo, hi = stats.t.interval(0.95, len(v) - 1, loc=mean, scale=sem)
        p_t = stats.ttest_1samp(v, 0.0).pvalue
        try: p_w = stats.wilcoxon(v).pvalue
        except ValueError: p_w = np.nan
        rows.append(dict(label=lab, n=len(v), mean=mean, ci95_low=lo, ci95_high=hi, sd=v.std(ddof=1),
                         p_ttest=p_t, p_wilcoxon=p_w))
    df = pd.DataFrame(rows).set_index("label")
    df["p_t_holm"] = holm(df.p_ttest.values)
    df["p_t_bh"] = bh(df.p_ttest.values)
    df["p_w_holm"] = holm(np.nan_to_num(df.p_wilcoxon.values, nan=1.0))
    df["sig_raw"] = df.p_ttest < ALPHA
    df["sig_holm"] = df.p_t_holm < ALPHA
    return df

tab = {
    "mdf_channel": slope_table(d_mdf, 0, ch_lbl),
    "rms_channel": slope_table(d_rms, 0, ch_lbl),
    "mdf_subject": slope_table(d_mdf, 1, sub_lbl),
    "rms_subject": slope_table(d_rms, 1, sub_lbl),
}
for k, df in tab.items():
    df.to_csv(os.path.join(OUT_DIR, f"slope_stats_{k}.csv"))
    print(f"\n{k}: mean slope, 95% CI, uncorrected / Holm / BH p-values")
    display(df.round(4))

# Overall effect: subject means (one value per subject) respect the dependence between channels
for n, M in [("ΔMDF (Hz/s)", d_mdf), ("ΔRMS (RMS units/s)", d_rms)]:
    sm = M.mean(1)
    lo, hi = stats.t.interval(0.95, S - 1, loc=sm.mean(), scale=stats.sem(sm))
    print(f"{n}: mean of subject means = {sm.mean():.4f}, 95% CI [{lo:.4f}, {hi:.4f}], p = {stats.ttest_1samp(sm, 0).pvalue:.2e}")

### 3.3 Trend Visualizations
Four views of the slopes: (i) one example MDF trajectory with its linear fit; (ii) 3-D bar charts of every subject × channel slope (bar colour = channel; ΔRMS bars point up when amplitude rises, ΔMDF bars point down when the spectrum shifts to lower frequencies); (iii) mean slopes per subject and per channel with 95% confidence intervals; (iv) p-values of the slope tests on a log scale.

In [ ]:
# Example trajectory: MDF of Subject 1, Channel 4 with the linear fit and a ±1.96 residual-SD band
s0, c0 = 0, 3
t = np.arange(1, E + 1); y = mdf_ep[s0, :, c0]
fit = ic_mdf[s0, c0] + d_mdf[s0, c0] * (t - 1)
band = 1.96 * sd_mdf[s0, c0]
plt.figure(figsize=(7, 3.5))
plt.plot(t, y, "b.", label="MDF per epoch")
plt.plot(t, fit, "k-", label=f"Linear fit (slope {d_mdf[s0, c0]:.3f} Hz/s, R² = {r2_mdf[s0, c0]:.2f})")
plt.plot(t, fit + band, "r--", lw=0.8, label="±1.96 residual SD"); plt.plot(t, fit - band, "r--", lw=0.8)
plt.xlabel("Time (s)"); plt.ylabel("Median frequency (Hz)")
plt.title("MDF trajectory with linear fit: Subject 1, Channel 4"); plt.legend(fontsize=8)
save_show("mdf_trajectory_sub1_ch4.png")

# Slope matrices (subjects × channels) as 3-D bar charts
def bar3d(M, title, zlabel, fname):
    fig = plt.figure(figsize=(10, 6)); ax = fig.add_subplot(111, projection="3d")
    xs, ys = np.meshgrid(np.arange(1, M.shape[0] + 1), np.arange(1, M.shape[1] + 1), indexing="ij")
    cols = plt.cm.viridis(np.tile(np.linspace(0, 1, M.shape[1]), (M.shape[0], 1))).reshape(-1, 4)
    ax.bar3d(xs.ravel() - 0.4, ys.ravel() - 0.4, np.zeros(M.size), 0.8, 0.8, M.ravel(), color=cols, shade=True)
    ax.set_xticks(np.arange(1, M.shape[0] + 1)); ax.set_yticks(np.arange(1, M.shape[1] + 1)); ax.tick_params(labelsize=7)
    ax.set_zlim(min(0, M.min()), max(0, M.max())); ax.view_init(elev=25, azim=-60)
    ax.set_xlabel("Subject"); ax.set_ylabel("Channel"); ax.set_zlabel(zlabel); ax.set_title(title)
    ch_cols_ = plt.cm.viridis(np.linspace(0, 1, M.shape[1]))
    ax.legend(handles=[Patch(color=c_, label=f"Ch{i + 1}") for i, c_ in enumerate(ch_cols_)], title="Channel", fontsize=7,
              loc="upper left", bbox_to_anchor=(1.02, 1.0))
    save_show(fname, bbox_inches="tight")

bar3d(d_rms, "ΔRMS per subject and channel", "Slope (RMS units/s)", "delta_rms_3d.png")
bar3d(d_mdf, "ΔMDF per subject and channel", "Slope (Hz/s)", "delta_mdf_3d.png")

# Mean slopes with 95% confidence intervals
def barci(ax, df, title, ylabel=None):
    x = np.arange(len(df)); err = np.vstack([df["mean"] - df.ci95_low, df.ci95_high - df["mean"]])
    ax.bar(x, df["mean"], color="#0072BD", yerr=err, capsize=3); ax.axhline(0, c="k", lw=0.6)
    ax.set_xticks(x); ax.set_xticklabels([l.replace("Sub", "").replace("Ch", "") for l in df.index], fontsize=8)
    ax.set_title(title, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel)

fig, ax = plt.subplots(2, 2, figsize=(12, 7))
barci(ax[0, 0], tab["mdf_subject"], "A) Mean ΔMDF per subject (across channels)", "Mean slope (Hz/s)")
barci(ax[0, 1], tab["mdf_channel"], "B) Mean ΔMDF per channel (across subjects)")
barci(ax[1, 0], tab["rms_subject"], "C) Mean ΔRMS per subject (across channels)", "Mean slope (RMS units/s)")
barci(ax[1, 1], tab["rms_channel"], "D) Mean ΔRMS per channel (across subjects)")
ax[1, 0].set_xlabel("Subject"); ax[1, 1].set_xlabel("Channel")
fig.suptitle("Mean slopes with 95% confidence intervals")
save_show("mean_slopes_ci.png")

# p-values of the slope tests (log scale)
def pplot(ax, df, title, ylabel=None):
    x = np.arange(len(df))
    ax.scatter(x, np.clip(df.p_ttest, 1e-12, 1), marker="o", c="#0072BD", label="t-test (uncorrected)")
    ax.scatter(x, np.clip(df.p_t_holm, 1e-12, 1), marker="D", facecolors="none", edgecolors="#D95319", label="t-test (Holm)")
    ax.scatter(x, np.clip(df.p_wilcoxon.fillna(1), 1e-12, 1), marker="x", c="k", label="Wilcoxon (uncorrected)")
    ax.axhline(ALPHA, c="r", ls="--", lw=0.8); ax.set_yscale("log"); ax.set_ylim(1e-8, 1.5)
    ax.set_xticks(x); ax.set_xticklabels([l.replace("Sub", "").replace("Ch", "") for l in df.index], fontsize=8)
    ax.set_title(title, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel)

fig, ax = plt.subplots(2, 2, figsize=(12, 7))
pplot(ax[0, 0], tab["mdf_subject"], "A) MDF slope tests per subject", "p-value")
pplot(ax[0, 1], tab["mdf_channel"], "B) MDF slope tests per channel")
pplot(ax[1, 0], tab["rms_subject"], "C) RMS slope tests per subject", "p-value")
pplot(ax[1, 1], tab["rms_channel"], "D) RMS slope tests per channel")
ax[1, 0].set_xlabel("Subject"); ax[1, 1].set_xlabel("Channel"); ax[0, 0].legend(fontsize=7)
fig.suptitle("p-values of slope tests (red dashed line: α = 0.05)")
save_show("slope_test_pvalues.png")

### 3.4 Summary of Temporal Trends
Channel- and subject-level pattern of the slope tests (uncorrected p-values; see the tables above for corrected values).

In [ ]:
mc, rc, ms, rs = tab["mdf_channel"], tab["rms_channel"], tab["mdf_subject"], tab["rms_subject"]

def names(df, mask):
    return ", ".join(sorted(df.index[mask], key=natural_key)) or "none"

trend_summary = pd.Series({
    "Channel with the most negative mean ΔMDF": f"{mc['mean'].idxmin()} ({mc['mean'].min():.4f} Hz/s)",
    "Channel with the least negative mean ΔMDF": f"{mc['mean'].idxmax()} ({mc['mean'].max():.4f} Hz/s)",
    "Channels with non-significant ΔMDF": names(mc, mc.p_ttest >= ALPHA),
    "Channel with the lowest mean ΔRMS": f"{rc['mean'].idxmin()} ({rc['mean'].min():.4f} RMS units/s)",
    "Channels with non-significant ΔRMS": names(rc, rc.p_ttest >= ALPHA),
    "Subjects with the two smallest |mean ΔMDF|": ", ".join(ms["mean"].abs().sort_values().index[:2]),
    "Subjects with non-significant ΔRMS": names(rs, rs.p_ttest >= ALPHA),
}, name="value").to_frame()
display(trend_summary)
trend_summary.to_csv(os.path.join(OUT_DIR, "temporal_trend_summary.csv"))

## 4. Spectral Feature Analysis
Four spectral indices are computed from each 1 s epoch power spectrum and compared with RMS through their linear trends. The *expected direction* is the sign of the slope under progressive fatigue (a downward spectral shift).

| Index | Definition | Expected slope |
|---|---|---|
| MDF | median frequency of the full spectrum | negative |
| MDF (notch-excluded) | median frequency after removing the 45–55 Hz bins around the 50 Hz notch | negative |
| log SMR | log10 of the spectral-moment ratio M₋₁ / M₅, with Mₖ = Σ f ᵏ P(f) for f ≥ 5 Hz | positive |
| log L/H | log10 of the ratio of power in 10–45 Hz to power in 55–95 Hz | positive |

**Output columns of the comparison table.** *% series, expected sign*: share of the 120 series whose slope has the expected sign; *% significant*: share that also has p < 0.05 in the within-series fit; *median within-series t*: median slope t statistic (expected direction positive); *median across-subject t*: median over channels of the one-sample t statistic across subjects; *channels significant (Holm)*: number of the 8 channels with a significant slope after Holm correction; *median R²*: median linear-fit R²; *effect size dz*: mean divided by SD of the signed slopes.

In [ ]:
NOTCH_BAND = (45, 55)   # Hz excluded for the notch-excluded MDF (data are notch-filtered at 50 Hz)
LOW_BAND = (10, 45)     # Hz, numerator band of the low/high power ratio
HIGH_BAND = (55, 95)    # Hz, denominator band (above the notch, below Nyquist = 100 Hz)
SMR_FMIN = 5.0          # Hz, lower bound for spectral moments (excludes DC and motion artefact)

INDEX_LABELS = {"rms": "RMS", "mdf": "MDF", "mdf_nr": "MDF (notch-excluded)", "logsmr": "log SMR",
                "loglh": "log L/H", "mav": "MAV", "wl": "WL", "zc": "ZC"}
EXPECTED = {"rms": +1, "mdf": -1, "mdf_nr": -1, "logsmr": +1, "loglh": +1}   # slope sign under fatigue
INDEX_ORDER = ["rms", "mdf", "mdf_nr", "logsmr", "loglh"]

def spectral_indices(segs):
    """segs: (E, FS, C) demeaned 1 s epochs -> dict of (E, C) index series."""
    f, P = psd_epochs(segs)
    out = {}
    out["mdf"] = median_freq(f, P, MDF_FMIN)
    keep = ~((f >= NOTCH_BAND[0]) & (f <= NOTCH_BAND[1]))
    out["mdf_nr"] = median_freq(f[keep], P[:, keep, :], MDF_FMIN)
    m = f >= SMR_FMIN
    ff, Pm = f[m][None, :, None], P[:, m, :]
    m_neg1, m_5 = (Pm * ff ** -1).sum(1), (Pm * ff ** 5).sum(1)
    out["logsmr"] = np.log10(np.maximum(m_neg1, 1e-12)) - np.log10(np.maximum(m_5, 1e-12))
    lo = (f >= LOW_BAND[0]) & (f <= LOW_BAND[1]); hi = (f >= HIGH_BAND[0]) & (f <= HIGH_BAND[1])
    out["loglh"] = np.log10(np.maximum(P[:, lo, :].sum(1), 1e-12)) - np.log10(np.maximum(P[:, hi, :].sum(1), 1e-12))
    return out

IDX = {k: np.zeros((S, E, N_CH)) for k in ["mdf", "mdf_nr", "logsmr", "loglh"]}
for s in range(S):
    segs = X[s].reshape(E, FS, N_CH)
    segs = segs - segs.mean(axis=1, keepdims=True)
    for k, v in spectral_indices(segs).items():
        IDX[k][s] = v
IDX["rms"] = rms_ep
IDX["mdf"] = mdf_ep                                  # use the Section 2 MDF series everywhere

def fit_stats(F):
    """Linear-fit slope, standard error, p-value and R² for every subject × channel series."""
    t = np.arange(F.shape[1]); o = {k: np.zeros((F.shape[0], F.shape[2])) for k in ["slope", "se", "p", "r2"]}
    for s in range(F.shape[0]):
        for c in range(F.shape[2]):
            r = stats.linregress(t, F[s, :, c])
            o["slope"][s, c], o["se"][s, c], o["p"][s, c], o["r2"][s, c] = r.slope, r.stderr, r.pvalue, r.rvalue ** 2
    return o

FIT = {k: fit_stats(IDX[k]) for k in INDEX_ORDER}

# Compare the indices through their slopes in the expected direction
rows = []
for name in INDEX_ORDER:
    e, F_ = EXPECTED[name], FIT[name]
    sl = e * F_["slope"]                                              # positive = expected direction
    t_series = sl / np.maximum(F_["se"], 1e-12)                       # within-series slope t statistic
    ch_tests = [stats.ttest_1samp(F_["slope"][:, c], 0.0) for c in range(N_CH)]
    ch_t = np.array([e * r.statistic for r in ch_tests]); ch_p = np.array([r.pvalue for r in ch_tests])
    rows.append(dict(index=INDEX_LABELS[name],
                     pct_expected_sign=100 * np.mean(sl > 0),
                     pct_sig_expected=100 * np.mean((sl > 0) & (F_["p"] < ALPHA)),
                     median_series_t=np.median(t_series),
                     median_channel_t=np.median(ch_t),
                     channels_sig_holm=int((holm(ch_p) < ALPHA).sum()),
                     median_R2=np.median(F_["r2"]),
                     effect_size_dz=sl.mean() / sl.std(ddof=1)))
idx_cmp = pd.DataFrame(rows).set_index("index")
display(idx_cmp.rename(columns={
    "pct_expected_sign": "% series, expected sign", "pct_sig_expected": "% significant, expected direction",
    "median_series_t": "median within-series t", "median_channel_t": "median across-subject t",
    "channels_sig_holm": "channels significant (Holm)", "median_R2": "median R²", "effect_size_dz": "effect size dz"}).round(3))
idx_cmp.to_csv(os.path.join(OUT_DIR, "spectral_index_comparison.csv"))

fig, ax = plt.subplots(1, 4, figsize=(16, 4.2))
panels = [("pct_expected_sign", "Series with expected slope sign (%)"),
          ("median_series_t", "Median within-series slope t"),
          ("median_channel_t", "Median across-subject t per channel"),
          ("median_R2", "Median R² of linear fit")]
for a, (col, ttl) in zip(ax, panels):
    a.bar(idx_cmp.index, idx_cmp[col], color=["#888", "#0072BD", "#D95319", "#77AC30", "#7E2F8E"])
    a.set_title(ttl, fontsize=9); plt.setp(a.get_xticklabels(), rotation=30, ha="right", fontsize=8)
save_show("spectral_index_comparison.png")

## 5. Fatigue-Zone Analysis
Following the JASA-style scheme, each subject × channel series is assigned to one of four zones from the signs of its RMS slope and spectral slope: *Fatigue* (RMS increasing, spectrum shifting downward), *Force increase* (RMS increasing, spectrum not shifting downward), *Recovery* (RMS decreasing, spectrum not shifting downward) and *Force decrease* (RMS decreasing, spectrum shifting downward). The zones are determined by slope signs only.

The fraction of a subject's channels in the Fatigue zone is used as an **exploratory fatigue-zone score**. The recordings contain no independent measure of fatigue (force, MVC or time to failure), so the score and the resulting ranking are descriptive and have not been validated against physiology. p-values from within-series linear fits assume independent epochs and are likewise descriptive.

### 5.1 Zone Assignment and Subject Scores
**Subject table columns.** *fatigue_score*: fraction of the 8 channels in the Fatigue zone; *confident_score*: fraction of channels in the Fatigue zone with both slopes individually significant (p < 0.05); *mean_d_spec* and *mean_d_rms*: mean spectral and RMS slope over channels; *n_fatigue*, *n_force_increase*, *n_recovery*, *n_force_decrease*: number of channels per zone; *spec_shift_strength*: mean spectral slope signed so that positive means a shift in the fatigue direction; *zone_rank*: rank by (fatigue_score, confident_score, spec_shift_strength), 1 = largest fatigue-zone score.

In [ ]:
JASA_SPEC = "mdf"       # spectral index used for the zone assignment
QUAD = ["Fatigue", "Force increase", "Recovery", "Force decrease"]
QCOL = ["#d62728", "#ff7f0e", "#2ca02c", "#1f77b4"]

def jasa_quadrants(slope_rms, slope_spec, spec_dir):
    rms_up = slope_rms > 0
    spec_fat = spec_dir * slope_spec > 0              # True: spectrum shifts in the fatigue direction
    return np.where(rms_up & spec_fat, 0, np.where(rms_up & ~spec_fat, 1, np.where(~rms_up & ~spec_fat, 2, 3)))

# Share of subject-channel series per zone for each spectral index
qshare = {}
for sp in ["mdf", "mdf_nr", "logsmr", "loglh"]:
    q = jasa_quadrants(FIT["rms"]["slope"], FIT[sp]["slope"], EXPECTED[sp])
    qshare[INDEX_LABELS[sp]] = [100 * np.mean(q == k) for k in range(4)]
qshare = pd.DataFrame(qshare, index=QUAD).T
print("Share of subject-channel series (%) per fatigue zone, by spectral index:")
display(qshare.round(1)); qshare.to_csv(os.path.join(OUT_DIR, "fatigue_zone_share_by_index.csv"))

# Zone assignment and subject scores with the selected spectral index
sp_dir = EXPECTED[JASA_SPEC]
Q = jasa_quadrants(FIT["rms"]["slope"], FIT[JASA_SPEC]["slope"], sp_dir)
conf = ((FIT["rms"]["slope"] > 0) & (FIT["rms"]["p"] < ALPHA) &
        (sp_dir * FIT[JASA_SPEC]["slope"] > 0) & (FIT[JASA_SPEC]["p"] < ALPHA))

subj = pd.DataFrame({
    "fatigue_score": (Q == 0).mean(1),                               # fraction of the 8 channels in the Fatigue zone
    "confident_score": conf.mean(1),                                 # fraction with both slopes individually significant
    "mean_d_spec": FIT[JASA_SPEC]["slope"].mean(1),
    "mean_d_rms": FIT["rms"]["slope"].mean(1),
}, index=sub_lbl)
for k, n in enumerate(QUAD):
    subj[f"n_{n.lower().replace(' ', '_')}"] = (Q == k).sum(1)
subj["spec_shift_strength"] = sp_dir * subj["mean_d_spec"]           # positive: shift in the fatigue direction
subj = subj.sort_values(["fatigue_score", "confident_score", "spec_shift_strength"], ascending=False)
subj["zone_rank"] = np.arange(1, S + 1)                              # 1 = highest fatigue-zone score
display(subj.round(3)); subj.to_csv(os.path.join(OUT_DIR, "subject_fatigue_zone_scores.csv"))

chan = pd.DataFrame({"frac_in_fatigue_zone": (Q == 0).mean(0), "frac_confident": conf.mean(0)}, index=ch_lbl)
print("Share of subjects in the Fatigue zone, per channel:")
display(chan.round(3)); chan.to_csv(os.path.join(OUT_DIR, "channel_fatigue_zone_fraction.csv"))

# Agreement with the ranking by mean ΔMDF. Both rankings are derived from the same slopes, so they are not independent.
mdf_slope_rank = pd.Series(d_mdf.mean(1), index=sub_lbl).rank(ascending=True)   # 1 = most negative mean ΔMDF
rho, p_rho = stats.spearmanr(mdf_slope_rank.loc[subj.index], subj["zone_rank"])
print(f"Spearman correlation between the fatigue-zone ranking and the mean-ΔMDF ranking: rho = {rho:.2f} (p = {p_rho:.3f}, n = {S})")


### 5.2 Fatigue-Zone Visualizations
Left: zone of every subject × channel series. Centre: number of channels per zone for each subject, sorted by fatigue-zone score. Right: fraction of subjects whose series falls in the Fatigue zone, per channel.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2), gridspec_kw={"width_ratios": [1.2, 1.2, 1]})
ax[0].imshow(Q.T, cmap=ListedColormap(QCOL), vmin=0, vmax=3, aspect="auto")
ax[0].set_xticks(range(S)); ax[0].set_xticklabels(range(1, S + 1), fontsize=8); ax[0].set_yticks(range(N_CH)); ax[0].set_yticklabels(range(1, N_CH + 1))
ax[0].grid(False); ax[0].set_xlabel("Subject"); ax[0].set_ylabel("Channel"); ax[0].set_title(f"Fatigue zone per subject-channel series ({INDEX_LABELS[JASA_SPEC]})")
ax[0].legend(handles=[Patch(color=c, label=n) for c, n in zip(QCOL, QUAD)], fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2)
order = subj.index
bottom = np.zeros(S)
for k in range(4):
    vals = np.array([subj.loc[o, f"n_{QUAD[k].lower().replace(' ', '_')}"] for o in order])
    ax[1].bar(range(S), vals, bottom=bottom, color=QCOL[k], label=QUAD[k]); bottom += vals
ax[1].set_xticks(range(S)); ax[1].set_xticklabels([o.replace("Sub", "") for o in order], fontsize=8)
ax[1].set_xlabel("Subject (sorted by fatigue-zone score, highest first)"); ax[1].set_ylabel("Number of channels"); ax[1].set_title("Zone counts per subject")
ax[2].bar(ch_lbl, chan["frac_in_fatigue_zone"], color="#d62728"); ax[2].set_ylim(0, 1)
ax[2].set_ylabel("Fraction of subjects"); ax[2].set_title("Fraction of subjects in the Fatigue zone, per channel"); ax[2].tick_params(axis="x", labelrotation=30)
save_show("fatigue_zone_analysis.png", bbox_inches="tight")

## 6. Temporal Trajectory Modeling
Three models are fitted to every subject × channel series of MDF and RMS: a straight line, a continuous two-segment piecewise-linear function (one breakpoint searched between `ONSET_TMIN` and `E − ONSET_TMIN`), and a saturating exponential. Models are compared with the corrected Akaike criterion (AICc); a non-linear model is preferred only if it improves AICc by at least `AIC_MARGIN` over the linear model. A breakpoint is reported as a valid *onset* only if the piecewise model is preferred over the linear model and the second segment moves in the fatigue direction more strongly than the first.

This analysis is exploratory. Better fit of a non-linear model describes the shape of the trajectory and does not identify a physiological mechanism.

### 6.1 Model Fitting and Summary
**Summary columns.** *% piecewise / exponential beats linear*: share of series where that model improves AICc by at least `AIC_MARGIN`; *% best non-linear*: share where a non-linear model is selected; *median R²*: median R² per model; *% valid onset*: share of series with a valid breakpoint-based onset; *onset median / q25 / q75*: distribution of valid onset times in seconds.

In [ ]:
ONSET_SIGNALS = ["mdf", "rms"]   # series analysed
ONSET_TMIN = 10                  # s, breakpoint search range [ONSET_TMIN, E - ONSET_TMIN]
AIC_MARGIN = 2.0                 # minimum AICc improvement over the linear model
USE_EXP = True                   # also fit the saturating exponential

def aicc(rss, n, k):
    rss = max(rss, 1e-12)
    return n * np.log(rss / n) + 2 * k + 2 * k * (k + 1) / (n - k - 1)

def _expf(t, a, b, tau):
    return a + b * (1 - np.exp(-t / tau))

def fit_models(y, exp_sign):
    n = len(y); t = np.arange(n, dtype=float)
    tss = np.sum((y - y.mean()) ** 2) + 1e-12
    res = {}
    # Linear model
    A = np.c_[np.ones(n), t]; c, *_ = np.linalg.lstsq(A, y, rcond=None); yh = A @ c; rss = np.sum((y - yh) ** 2)
    res.update(aicc_lin=aicc(rss, n, 2), r2_lin=1 - rss / tss, yh_lin=yh, slope_lin=c[1])
    # Continuous two-segment piecewise-linear model (breakpoint chosen by least squares)
    best = (np.inf, None, None, None)
    for tau in range(ONSET_TMIN, n - ONSET_TMIN):
        B = np.c_[np.ones(n), t, np.maximum(0, t - tau)]
        cc, *_ = np.linalg.lstsq(B, y, rcond=None); yy = B @ cc; r = np.sum((y - yy) ** 2)
        if r < best[0]: best = (r, tau, cc, yy)
    r, tau, cc, yy = best
    res.update(aicc_pw=aicc(r, n, 4), r2_pw=1 - r / tss, yh_pw=yy, tau=float(tau), slope1=cc[1], slope2=cc[1] + cc[2])
    # Saturating exponential
    res.update(aicc_exp=np.nan, r2_exp=np.nan, yh_exp=np.full(n, np.nan))
    if USE_EXP:
        try:
            p0 = [y[:5].mean(), y[-5:].mean() - y[:5].mean(), n / 3]
            popt, _ = curve_fit(_expf, t, y, p0=p0, bounds=([-np.inf, -np.inf, 1.0], [np.inf, np.inf, 10.0 * n]), maxfev=5000)
            yy = _expf(t, *popt); r = np.sum((y - yy) ** 2)
            res.update(aicc_exp=aicc(r, n, 3), r2_exp=1 - r / tss, yh_exp=yy)
        except Exception:
            pass
    # Model choice and onset validity
    cand = {"linear": res["aicc_lin"], "piecewise": res["aicc_pw"], "exponential": res["aicc_exp"]}
    best_nl = min([k for k in ["piecewise", "exponential"] if not np.isnan(cand[k])], key=lambda k: cand[k])
    res["best_model"] = best_nl if cand["linear"] - cand[best_nl] >= AIC_MARGIN else "linear"
    pw_beats_lin = (res["aicc_lin"] - res["aicc_pw"]) >= AIC_MARGIN
    res["valid_onset"] = bool(pw_beats_lin and exp_sign * res["slope2"] > 0 and exp_sign * res["slope2"] > exp_sign * res["slope1"])
    return res

recs, CURVES = [], {}
for sig in ONSET_SIGNALS:
    for s in range(S):
        for c in range(N_CH):
            r = fit_models(IDX[sig][s, :, c], EXPECTED[sig])
            CURVES[(sig, s, c)] = {k: r[k] for k in ["yh_lin", "yh_pw", "yh_exp"]}
            recs.append(dict(signal=sig, subject=sub_lbl[s], channel=ch_lbl[c], s=s, c=c,
                             **{k: v for k, v in r.items() if not k.startswith("yh_")}))
fits = pd.DataFrame(recs)
fits["onset_s"] = np.where(fits.valid_onset, fits.tau, np.nan)
fits.drop(columns=["s", "c"]).to_csv(os.path.join(OUT_DIR, "trajectory_model_fits.csv"), index=False)

summ = []
for sig in ONSET_SIGNALS:
    d = fits[fits.signal == sig]
    summ.append(dict(signal=INDEX_LABELS[sig],
                     pct_piecewise_beats_linear=100 * np.mean((d.aicc_lin - d.aicc_pw) >= AIC_MARGIN),
                     pct_exponential_beats_linear=100 * np.mean((d.aicc_lin - d.aicc_exp) >= AIC_MARGIN) if USE_EXP else np.nan,
                     pct_best_nonlinear=100 * np.mean(d.best_model != "linear"),
                     median_R2_linear=d.r2_lin.median(), median_R2_piecewise=d.r2_pw.median(), median_R2_exponential=d.r2_exp.median(),
                     pct_valid_onset=100 * d.valid_onset.mean(),
                     onset_median_s=d.onset_s.median(), onset_q25=d.onset_s.quantile(0.25), onset_q75=d.onset_s.quantile(0.75)))
summ = pd.DataFrame(summ).set_index("signal")
display(summ.rename(columns={
    "pct_piecewise_beats_linear": "% piecewise beats linear", "pct_exponential_beats_linear": "% exponential beats linear",
    "pct_best_nonlinear": "% best model non-linear", "median_R2_linear": "median R² linear", "median_R2_piecewise": "median R² piecewise",
    "median_R2_exponential": "median R² exponential", "pct_valid_onset": "% valid onset",
    "onset_median_s": "onset median (s)", "onset_q25": "onset Q25 (s)", "onset_q75": "onset Q75 (s)"}).round(2)); summ.to_csv(os.path.join(OUT_DIR, "trajectory_model_summary.csv"))


### 6.2 Example Fit and Onset Times
Left: the three models fitted to one example series. Right: breakpoint-based onset times of the series with a valid onset (blank cells: no valid onset), and their distribution.

In [ ]:
# Example fit (Subject 1, Channel 4, MDF)
ex_sig, ex_s, ex_c = "mdf", 0, 3
tt = np.arange(1, E + 1); cv = CURVES[(ex_sig, ex_s, ex_c)]
row = fits[(fits.signal == ex_sig) & (fits.s == ex_s) & (fits.c == ex_c)].iloc[0]
plt.figure(figsize=(7.5, 3.8))
plt.plot(tt, IDX[ex_sig][ex_s, :, ex_c], "k.", ms=4, label="MDF per epoch")
plt.plot(tt, cv["yh_lin"], "b-", label=f"Linear (R² = {row.r2_lin:.2f})")
plt.plot(tt, cv["yh_pw"], "r-", label=f"Piecewise (R² = {row.r2_pw:.2f}, breakpoint {row.tau:.0f} s)")
if USE_EXP: plt.plot(tt, cv["yh_exp"], "g-", label=f"Exponential (R² = {row.r2_exp:.2f})")
plt.axvline(row.tau + 1, c="r", ls=":"); plt.xlabel("Time (s)"); plt.ylabel("Median frequency (Hz)")
plt.title(f"Trajectory model comparison: {sub_lbl[ex_s]}, {ch_lbl[ex_c]} (preferred: {row.best_model})"); plt.legend(fontsize=8)
save_show("trajectory_example_fits.png")

# Onset times (valid onsets only)
fig, ax = plt.subplots(1, len(ONSET_SIGNALS) + 1, figsize=(5 * (len(ONSET_SIGNALS) + 1), 3.8))
ONSET = {}
for a, sig in zip(ax, ONSET_SIGNALS):
    M = np.full((S, N_CH), np.nan)
    d = fits[fits.signal == sig]
    M[d.s.values, d.c.values] = d.onset_s.values
    ONSET[sig] = M
    im = a.imshow(M.T, cmap="viridis", aspect="auto", vmin=ONSET_TMIN, vmax=E - ONSET_TMIN)
    a.grid(False)
    a.set_xticks(range(S)); a.set_xticklabels(range(1, S + 1), fontsize=8); a.set_yticks(range(N_CH)); a.set_yticklabels(range(1, N_CH + 1))
    a.set_xlabel("Subject"); a.set_ylabel("Channel"); a.set_title(f"Onset time (s): {INDEX_LABELS[sig]} (blank: no valid onset)"); plt.colorbar(im, ax=a)
for sig in ONSET_SIGNALS:
    ax[-1].hist(fits[fits.signal == sig].onset_s.dropna(), bins=np.arange(ONSET_TMIN, E - ONSET_TMIN + 5, 5), alpha=0.6, label=INDEX_LABELS[sig])
ax[-1].set_xlabel("Onset time (s)"); ax[-1].set_ylabel("Number of series"); ax[-1].set_title("Distribution of valid onset times"); ax[-1].legend()
save_show("trajectory_onset_times.png")

## 7. Exploratory Fatigue-Response Ranking
The subjects are ranked on three exploratory criteria: the fatigue-zone score (Section 5) and the median valid onset time of MDF and of RMS (Section 6; later onset = smaller fatigue response). The composite is the mean of the available ranks; rank 1 denotes the smallest EMG-derived fatigue response in this recording. Because the dataset has no independent fatigue ground truth, this ranking is **exploratory and not a validated measure of fatigue susceptibility**.

**Table columns.** *rank_zone_score*, *rank_onset_mdf*, *rank_onset_rms*: rank of the subject on each criterion (1 = smallest response); *median_onset_*_s*: median valid onset time in seconds (NaN if the subject has no valid onset); *n_valid_onsets_**: number of channels with a valid onset; *composite_rank_mean*: mean of the available ranks; *composite_order*: final order (1 = smallest response).

In [ ]:
ranking = subj[["fatigue_score", "confident_score", "zone_rank"]].copy().loc[sub_lbl]
for sig in ONSET_SIGNALS:
    ranking[f"median_onset_{sig}_s"] = np.nanmedian(ONSET[sig], axis=1)
    ranking[f"n_valid_onsets_{sig}"] = np.sum(~np.isnan(ONSET[sig]), axis=1)
ranking["rank_zone_score"] = ranking.fatigue_score.rank(ascending=True, method="average")                   # low score = smaller response = rank 1
for sig in ONSET_SIGNALS:
    ranking[f"rank_onset_{sig}"] = ranking[f"median_onset_{sig}_s"].rank(ascending=False, method="average")  # late onset = rank 1
rank_cols = ["rank_zone_score"] + [f"rank_onset_{s_}" for s_ in ONSET_SIGNALS]
ranking["composite_rank_mean"] = ranking[rank_cols].mean(axis=1, skipna=True)
ranking["composite_order"] = ranking.composite_rank_mean.rank(method="min").astype(int)                    # 1 = smallest response
ranking = ranking.sort_values("composite_order")
display(ranking.round(2)); ranking.to_csv(os.path.join(OUT_DIR, "fatigue_response_ranking.csv"))

## 8. Feature Extraction for Window Classification
In addition to RMS and the spectral indices of Section 4, three time-domain features are computed per 1 s epoch (after removing the epoch mean):
- **MAV:** mean absolute value.
- **WL:** waveform length, the summed absolute difference between successive samples.
- **ZC:** zero crossings, counted only when the amplitude step between the two samples is at least 1 unit (a dead band appropriate for the 8-bit data).

Three feature sets are compared. Each feature is computed for all 8 channels.

In [ ]:
IDX_C = dict(IDX)
mav = np.zeros((S, E, N_CH)); wl = np.zeros((S, E, N_CH)); zc = np.zeros((S, E, N_CH))
for s in range(S):
    segs = X[s].reshape(E, FS, N_CH)
    segs = segs - segs.mean(axis=1, keepdims=True)
    mav[s] = np.abs(segs).mean(axis=1)
    dif = np.diff(segs, axis=1)
    wl[s] = np.abs(dif).sum(axis=1)
    zc[s] = ((segs[:, :-1] * segs[:, 1:] < 0) & (np.abs(dif) >= 1.0)).sum(axis=1)
IDX_C.update(mav=mav, wl=wl, zc=zc)
assert all(np.isfinite(v).all() for v in IDX_C.values()), "Non-finite epoch feature values."

FEATURE_SETS = {
    "Core":           ["rms", "mdf"],
    "Spectral-index": ["rms", "mdf_nr", "logsmr", "loglh"],
    "Combined":       ["rms", "mdf", "mdf_nr", "logsmr", "loglh", "mav", "wl", "zc"],
}
LOG_FEATS = {"logsmr", "loglh"}   # log-scale features: normalized by subtraction; all others by relative change

feature_set_table = pd.DataFrame({
    "features per channel": [", ".join(INDEX_LABELS[f] for f in feats) for feats in FEATURE_SETS.values()],
    "total features": [len(feats) * N_CH for feats in FEATURE_SETS.values()],
}, index=pd.Index(list(FEATURE_SETS), name="feature set"))
display(feature_set_table)
FS_DISPLAY = {k: f"{k} ({len(v) * N_CH} features)" for k, v in FEATURE_SETS.items()}

## 9. Window Dataset and Subject-Wise Evaluation Framework
### 9.1 Configuration and Models
All settings below were fixed before evaluation and are not tuned on the results. Standardization is part of the logistic-regression pipeline, so it is re-fitted on the training subjects of every LOSO fold.

In [ ]:
WIN_S = 5               # window length (s): mean of 5 consecutive 1 s epoch features
STEP_S = 1              # window step (s)
LABEL_FRAC = 0.25       # first / last 25% of the recording = early / late (binary task)
BASE_S = 10             # initial segment (s) used as the per-subject baseline; windows starting inside it are excluded
N_TREES = 200
N_PERM = 1000           # label permutations for the permutation test (smallest possible p = 1 / (N_PERM + 1))
EVAL_FEATURE_SET = "Combined"          # feature set for the permutation test and channel analysis (fixed a priori)
EVAL_MODEL = "Logistic Regression"     # model for the permutation test, channel analysis and sensitivity analysis

MODELS = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=1.0)),
    "Random Forest":       RandomForestClassifier(n_estimators=N_TREES, min_samples_leaf=2, n_jobs=-1, random_state=SEED),
    "Gradient Boosting":   HistGradientBoostingClassifier(max_depth=3, learning_rate=0.1, max_iter=150, random_state=SEED),
}

### 9.2 Window Construction, Baseline Normalization and LOSO Evaluation
- `build_dataset` averages the epoch features over each window, optionally expresses them relative to the subject's own baseline, assigns the window label, and returns the feature matrix, labels, subject indices and column names.
- The baseline uses only the first `BASE_S` seconds of the same subject; no information from other subjects or from later in the recording enters the normalization. Windows starting inside the baseline segment are always excluded, so the baseline never appears in an evaluated window; this applies equally to raw features so that raw and normalized results use identical windows.
- `loso` fits the model on 14 subjects and predicts the held-out subject, returning per-subject accuracy, balanced accuracy, F1 and (binary task) AUC.
- `ci95` is a percentile bootstrap over subjects (one value per held-out subject).

In [ ]:
def window_label(t0, W, task, frac):
    """Label of the window starting at epoch t0; -1 marks windows excluded from the task."""
    if task == "binary":
        n = int(round(frac * E))
        if t0 + W <= n: return 0          # early
        if t0 >= E - n: return 1          # late
        return -1
    b = E // 3
    for k in range(3):
        if t0 >= k * b and t0 + W <= (k + 1) * b: return k      # early / middle / late
    return -1

def build_dataset(feats, task="binary", W=None, step=None, frac=None, normalize=True):
    """Return X (windows × features), labels, subject index per window and feature column names."""
    W = W or WIN_S; step = step or STEP_S; frac = frac or LABEL_FRAC
    cols = [f"{f}|{ch}" for ch in ch_lbl for f in feats]                 # channel-major column order
    Xs, ys, gs = [], [], []
    for s in range(S):
        A = np.stack([IDX_C[f][s] for f in feats], axis=-1).astype(float)   # (E, C, F)
        if normalize:
            base = A[:BASE_S].mean(0)                                      # per-subject baseline, shape (C, F)
            for j, f in enumerate(feats):
                if f in LOG_FEATS:
                    A[:, :, j] = A[:, :, j] - base[:, j]
                else:
                    if base[:, j].min() <= 0:
                        raise ValueError(f"Non-positive baseline for '{f}' in subject {s + 1}: relative change is undefined.")
                    A[:, :, j] = A[:, :, j] / base[:, j] - 1
        for t0 in range(0, E - W + 1, step):
            if t0 < BASE_S: continue                                       # window overlaps the baseline segment
            lab = window_label(t0, W, task, frac)
            if lab < 0: continue
            Xs.append(A[t0:t0 + W].mean(0).ravel()); ys.append(lab); gs.append(s)
    Xf, y, g = np.array(Xs), np.array(ys), np.array(gs)
    assert Xf.shape[0] == len(y) == len(g), "Feature, label and subject arrays are misaligned."
    assert np.isfinite(Xf).all(), "Non-finite values in the window feature matrix."
    assert all(len(np.unique(y[g == s])) == len(np.unique(y)) for s in np.unique(g)), "A subject is missing a class."
    return Xf, y, g, cols

def loso(model, Xf, y, groups):
    """Leave-one-subject-out evaluation. Returns per-subject metrics and the pooled true / predicted labels."""
    rows_, yt, yp = [], [], []
    k = len(np.unique(y))
    for tr, te in LeaveOneGroupOut().split(Xf, y, groups):
        m = clone(model).fit(Xf[tr], y[tr]); p = m.predict(Xf[te])
        r = dict(subject=int(groups[te][0]), acc=accuracy_score(y[te], p), bacc=balanced_accuracy_score(y[te], p))
        if k == 2:
            r["f1"] = f1_score(y[te], p, zero_division=0)
            try: r["auc"] = roc_auc_score(y[te], m.predict_proba(Xf[te])[:, 1])
            except Exception: r["auc"] = np.nan
        else:
            r["f1"] = f1_score(y[te], p, average="macro", zero_division=0); r["auc"] = np.nan
        rows_.append(r); yt.append(y[te]); yp.append(p)
    return pd.DataFrame(rows_), np.concatenate(yt), np.concatenate(yp)

def ci95(v, n_boot=10000, seed=SEED):
    """95% percentile bootstrap confidence interval of the mean over subjects."""
    v = np.asarray(v, float); v = v[~np.isnan(v)]
    rng = np.random.default_rng(seed)
    boot = v[rng.integers(0, len(v), (n_boot, len(v)))].mean(1)
    return tuple(np.percentile(boot, [2.5, 97.5]))

# Evaluation dataset (EVAL_FEATURE_SET, baseline-normalized); reused in Sections 10.4, 13 and 14
Xb, yb, gb, cb = build_dataset(FEATURE_SETS[EVAL_FEATURE_SET], "binary")
MAJORITY_RATE = np.bincount(yb).max() / len(yb)      # accuracy of always predicting the larger class
print(f"Binary task: {len(yb)} windows from {len(np.unique(gb))} subjects | early/late windows {np.bincount(yb)} | "
      f"windows per subject {np.bincount(gb).min()}-{np.bincount(gb).max()} | features {Xb.shape[1]}")
print(f"Majority-class accuracy: {MAJORITY_RATE:.3f} (balanced accuracy and AUC chance level: 0.5)")

## 10. Binary Temporal-Window Classification
Early versus late windows (Section 9.2), evaluated with LOSO for every feature set and model, with and without baseline normalization. A single-feature reference classifier (logistic regression on the channel-averaged normalized or raw MDF) is included for comparison. Results are summarized across the 15 held-out subjects.
### 10.1 Subject-Wise Evaluation: Raw and Baseline-Normalized Features

In [ ]:
BASELINE_SET = "MDF-only (channel mean, 1 feature)"
BASELINE_MODEL = "Logistic Regression (single feature)"

results = []
for normalize in [True, False]:
    for fs_name, feats in FEATURE_SETS.items():
        Xf, y, g, _ = build_dataset(feats, "binary", normalize=normalize)
        for m_name, model in MODELS.items():
            df, _, _ = loso(model, Xf, y, g)
            df.insert(0, "model", m_name); df.insert(0, "feature_set", fs_name); df.insert(0, "baseline_normalized", normalize)
            results.append(df)
    # Single-feature reference: channel-averaged MDF with a learned threshold
    Xf, y, g, _ = build_dataset(["mdf"], "binary", normalize=normalize)
    df, _, _ = loso(MODELS["Logistic Regression"], Xf.mean(axis=1, keepdims=True), y, g)
    df.insert(0, "model", BASELINE_MODEL); df.insert(0, "feature_set", BASELINE_SET); df.insert(0, "baseline_normalized", normalize)
    results.append(df)
res = pd.concat(results, ignore_index=True)
res.to_csv(os.path.join(OUT_DIR, "binary_per_subject_results.csv"), index=False)

def summarise(d):
    """Across-subject mean, SD and bootstrap CI of every metric per configuration."""
    key = ["baseline_normalized", "feature_set", "model"]
    out = d.groupby(key, sort=False).agg(acc_mean=("acc", "mean"), acc_sd=("acc", "std"), bacc_mean=("bacc", "mean"),
                                         f1_mean=("f1", "mean"), auc_mean=("auc", "mean")).reset_index()
    cis = [ci95(d[(d.baseline_normalized == r.baseline_normalized) & (d.feature_set == r.feature_set) & (d.model == r.model)].acc)
           for r in out.itertuples()]
    out["acc_ci95_low"] = [c[0] for c in cis]; out["acc_ci95_high"] = [c[1] for c in cis]
    return out

summ_c = summarise(res)
summ_c.to_csv(os.path.join(OUT_DIR, "binary_summary.csv"), index=False)

### 10.2 Accuracy by Feature Set and Model

In [ ]:
labels = ["MDF-only\n(1 feature)"] + [FS_DISPLAY[k].replace(" (", "\n(") for k in FEATURE_SETS]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2), sharey=True)
for a, norm in zip(ax, [True, False]):
    d = summ_c[summ_c.baseline_normalized == norm]
    w = 0.2; xs = np.arange(len(labels))
    b0 = d[d.feature_set == BASELINE_SET]
    a.bar(0, b0.acc_mean.iloc[0], w * 3, yerr=b0.acc_sd.iloc[0], color="#888", capsize=3, label="MDF-only reference")
    for i, m_name in enumerate(MODELS):
        vals = [d[(d.feature_set == fsn) & (d.model == m_name)] for fsn in FEATURE_SETS]
        a.bar(np.arange(1, 4) + (i - 1) * w, [v.acc_mean.iloc[0] for v in vals], w, yerr=[v.acc_sd.iloc[0] for v in vals], capsize=2, label=m_name)
    a.axhline(MAJORITY_RATE, c="r", ls="--", lw=0.8, label=f"Majority-class rate ({MAJORITY_RATE:.2f})")
    a.set_xticks(xs); a.set_xticklabels(labels, fontsize=8)
    a.set_title(("Baseline-normalized" if norm else "Raw") + " features", fontsize=10)
ax[0].set_ylabel("Accuracy (mean ± SD across subjects)"); ax[0].legend(fontsize=8); ax[0].set_ylim(0.3, 1.05)
fig.suptitle("Binary early-vs-late temporal-window classification (LOSO)")
save_show("binary_accuracy_by_feature_set.png")

### 10.3 Paired Subject-Level Comparisons
Wilcoxon signed-rank tests on per-subject accuracies (n = 15 held-out subjects) compare feature sets, baseline normalization and the single-feature reference, with Holm correction across all comparisons. These tests are descriptive.

In [ ]:
def paired(a_cfg, b_cfg, label):
    """a_cfg / b_cfg = (baseline_normalized, feature_set, model); compares per-subject accuracy."""
    A = res[(res.baseline_normalized == a_cfg[0]) & (res.feature_set == a_cfg[1]) & (res.model == a_cfg[2])].sort_values("subject").acc.values
    B = res[(res.baseline_normalized == b_cfg[0]) & (res.feature_set == b_cfg[1]) & (res.model == b_cfg[2])].sort_values("subject").acc.values
    try: p = stats.wilcoxon(A, B).pvalue
    except ValueError: p = np.nan
    return dict(comparison=label, mean_diff=np.mean(A - B), p_wilcoxon=p, subjects_better=int((A > B).sum()), subjects_worse=int((A < B).sum()))

pc = []
for m_name in MODELS:
    pc.append(paired((True, "Spectral-index", m_name), (True, "Core", m_name), f"{m_name}: Spectral-index vs Core"))
    pc.append(paired((True, "Combined", m_name), (True, "Core", m_name), f"{m_name}: Combined vs Core"))
    pc.append(paired((True, "Core", m_name), (False, "Core", m_name), f"{m_name}: normalized vs raw (Core)"))
    pc.append(paired((True, "Core", m_name), (True, BASELINE_SET, BASELINE_MODEL), f"{m_name}: Core vs MDF-only reference"))
pc = pd.DataFrame(pc); pc["p_holm"] = holm(np.nan_to_num(pc.p_wilcoxon.values, nan=1.0))
display(pc.round(4)); pc.to_csv(os.path.join(OUT_DIR, "binary_paired_comparisons.csv"), index=False)

### 10.4 Permutation Test
The test asks whether the observed LOSO balanced accuracy exceeds what is obtained when the relation between features and labels is destroyed. Labels are shuffled **within each subject** (keeping each subject's class balance and the subject structure), the model is trained on the permuted training labels, and it is scored on the held-out subject's true labels. The p-value is `(1 + #{null ≥ observed}) / (1 + N_PERM)`. A subject-level exact sign-flip test on the per-subject balanced accuracy minus 0.5 (2¹⁵ sign patterns) is reported alongside. Both tests show that performance exceeds chance; because the labels are defined by recording time, they do not show that the classifier is specific to fatigue.

In [ ]:
def loso_perm_null(model, Xf, y, groups, n_perm, seed=SEED):
    """Null distribution of the mean LOSO balanced accuracy under within-subject label permutation."""
    rng = np.random.default_rng(seed)
    folds = list(LeaveOneGroupOut().split(Xf, y, groups))
    subj_idx = [np.where(groups == g)[0] for g in np.unique(groups)]
    null = np.empty(n_perm)
    for b in range(n_perm):
        y_perm = y.copy()
        for idx in subj_idx:
            y_perm[idx] = rng.permutation(y[idx])
        null[b] = np.mean([balanced_accuracy_score(y[te], clone(model).fit(Xf[tr], y_perm[tr]).predict(Xf[te]))
                           for tr, te in folds])
    return null

# Observed per-subject results of the pre-specified configuration (already computed in Section 10.1)
eval_cfg = res.baseline_normalized & (res.feature_set == EVAL_FEATURE_SET) & (res.model == EVAL_MODEL)
obs_df = res[eval_cfg].sort_values("subject")
obs = obs_df.bacc.mean()
null = loso_perm_null(MODELS[EVAL_MODEL], Xb, yb, gb, N_PERM)
p_perm = (1 + np.sum(null >= obs)) / (1 + N_PERM)

d = obs_df.bacc.values - 0.5
flips = np.array(list(itertools.product([-1, 1], repeat=len(d))))
p_flip = np.mean((flips * d).mean(1) >= d.mean() - 1e-12)

print(f"Configuration: {EVAL_FEATURE_SET} features, {EVAL_MODEL}, baseline-normalized")
print(f"Observed LOSO balanced accuracy = {obs:.3f} | null mean = {null.mean():.3f}, null 95th percentile = {np.percentile(null, 95):.3f}")
print(f"Within-subject label permutation test: p = {p_perm:.4f} (N_PERM = {N_PERM}; smallest attainable p = {1 / (1 + N_PERM):.4f})")
print(f"Subject-level sign-flip test: p = {p_flip:.5f} (n = {len(d)} subjects)")
pd.DataFrame({"null_balanced_accuracy": null}).to_csv(os.path.join(OUT_DIR, "permutation_null_distribution.csv"), index=False)

## 11. Three-Class Temporal-Window Classification
The recording is divided into three equal parts (E // 3 s each). Windows lying entirely within the first, second or third part are labeled *early*, *middle* and *late*; windows that straddle a boundary or start inside the baseline segment are excluded. Features are baseline-normalized. The task is harder than the binary one because the middle class is adjacent to both others; the results describe discrimination of temporal stages of the recording. Chance level for balanced accuracy and macro F1 is about 1/3.

In [ ]:
res3 = []
for fs_name, feats in FEATURE_SETS.items():
    Xf3, y3, g3, _ = build_dataset(feats, "three", normalize=True)
    for m_name, model in MODELS.items():
        d, yt3, yp3 = loso(model, Xf3, y3, g3)
        res3.append(dict(feature_set=fs_name, model=m_name, acc_mean=d.acc.mean(), acc_sd=d.acc.std(),
                         bacc_mean=d.bacc.mean(), macro_f1=d.f1.mean()))
print(f"Three-class task: {len(y3)} windows | early/middle/late windows {np.bincount(y3)}")
res3 = pd.DataFrame(res3)
res3.to_csv(os.path.join(OUT_DIR, "three_class_summary.csv"), index=False)
display(res3.rename(columns={"feature_set": "Feature set", "model": "Model", "acc_mean": "Accuracy", "acc_sd": "Accuracy SD",
                             "bacc_mean": "Balanced accuracy", "macro_f1": "Macro F1"}).round(3))

# Confusion matrix of the configuration with the highest macro F1 (pooled over held-out subjects, row-normalized)
b3 = res3.loc[res3.macro_f1.idxmax()]
Xf3, y3, g3, _ = build_dataset(FEATURE_SETS[b3.feature_set], "three", normalize=True)
_, yt3, yp3 = loso(MODELS[b3.model], Xf3, y3, g3)
fig, ax = plt.subplots(figsize=(4.8, 4.2))
ConfusionMatrixDisplay(confusion_matrix(yt3, yp3, normalize="true"), display_labels=["Early", "Middle", "Late"]).plot(
    ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
ax.grid(False); ax.set_xlabel("Predicted window label"); ax.set_ylabel("True window label")
ax.set_title(f"Three-class temporal-window classification\n{b3.feature_set} features, {b3.model} (highest macro F1)", fontsize=9)
save_show("three_class_confusion_matrix.png")

## 12. Sensitivity Analysis
The binary classification is repeated for different window lengths and label fractions with the Core and Combined feature sets (logistic regression, baseline-normalized features). All configurations are reported; the purpose is to assess whether the conclusions depend on these choices, not to select the best one. Balanced accuracy is shown because the number of early and late windows differs when the label fraction or window length changes.

In [ ]:
sens = []
for W in [3, 5, 10]:
    for frac in [0.20, 0.25, 0.33]:
        for fs_name in ["Core", "Combined"]:
            Xs_, ys_, gs_, _ = build_dataset(FEATURE_SETS[fs_name], "binary", W=W, frac=frac, normalize=True)
            d, _, _ = loso(MODELS[EVAL_MODEL], Xs_, ys_, gs_)
            sens.append(dict(window_s=W, label_frac=frac, feature_set=fs_name, acc_mean=d.acc.mean(), bacc_mean=d.bacc.mean(),
                             acc_sd=d.acc.std(), n_windows=len(ys_)))
sens = pd.DataFrame(sens)
sens.to_csv(os.path.join(OUT_DIR, "sensitivity_analysis.csv"), index=False)
print(f"Balanced accuracy ({EVAL_MODEL}, baseline-normalized), mean over held-out subjects:")
display(sens.pivot_table(index=["window_s", "label_frac"], columns="feature_set", values="bacc_mean")
        .rename_axis(index=["Window length (s)", "Label fraction"], columns="Feature set").round(3))

## 13. Channel Contribution Analysis
Three descriptive views of the contribution of individual channels (Combined features, logistic regression, baseline-normalized): classification with a single channel, classification with one channel removed, and classification with the top-k channels. For the top-k analysis the channels are ranked **inside each LOSO fold from the training subjects only** (mean absolute Welch t statistic between classes), so the held-out subject never influences channel selection.

In [ ]:
ch_model = MODELS[EVAL_MODEL]
Xc, yc, gc, cols_c = Xb, yb, gb, cb               # evaluation dataset from Section 9.2
ch_cols = [np.array([j for j, nm in enumerate(cols_c) if nm.endswith("|" + ch)]) for ch in ch_lbl]
all_acc = obs_df.acc.mean()                         # all-channel accuracy (Section 10.1 result)

ch_res = []
for c in range(N_CH):
    single = loso(ch_model, Xc[:, ch_cols[c]], yc, gc)[0].acc
    keep = np.concatenate([ch_cols[k] for k in range(N_CH) if k != c])
    dropped = loso(ch_model, Xc[:, keep], yc, gc)[0].acc
    ch_res.append(dict(channel=ch_lbl[c], single_channel_acc=single.mean(), single_channel_sd=single.std(),
                       drop_one_acc=dropped.mean(), change_when_dropped=dropped.mean() - all_acc))
ch_res = pd.DataFrame(ch_res).set_index("channel")
print(f"Accuracy with all channels: {all_acc:.3f}")
display(ch_res.round(3)); ch_res.to_csv(os.path.join(OUT_DIR, "channel_single_and_drop_one.csv"))
print("Channels ranked by single-channel accuracy (best first):", ch_res.single_channel_acc.sort_values(ascending=False).index.tolist())

def channel_scores(Xtr, ytr):
    """Channel relevance from training data only: mean |Welch t| of the channel's features between classes."""
    t_ = stats.ttest_ind(Xtr[ytr == 1], Xtr[ytr == 0], equal_var=False, axis=0).statistic
    return np.array([np.nanmean(np.abs(t_[cols])) for cols in ch_cols])

ks = list(range(1, N_CH + 1)); topk = {k: [] for k in ks}; sel = np.zeros((S, N_CH))
for tr, te in LeaveOneGroupOut().split(Xc, yc, gc):
    order = np.argsort(-channel_scores(Xc[tr], yc[tr])); sel[int(gc[te][0]), order[:3]] = 1
    for k in ks:
        cols_k = np.concatenate([ch_cols[c] for c in order[:k]])
        m = clone(ch_model).fit(Xc[tr][:, cols_k], yc[tr]); topk[k].append(accuracy_score(yc[te], m.predict(Xc[te][:, cols_k])))
topk_df = pd.DataFrame({"k": ks, "acc_mean": [np.mean(topk[k]) for k in ks], "acc_sd": [np.std(topk[k], ddof=1) for k in ks]})
display(topk_df.round(3)); topk_df.to_csv(os.path.join(OUT_DIR, "topk_channel_accuracy.csv"), index=False)
print("Number of folds (of 15) in which each channel ranked in the in-fold top 3:", dict(zip(ch_lbl, sel.sum(0).astype(int))))

fig, ax = plt.subplots(1, 3, figsize=(15, 3.9))
ax[0].bar(ch_lbl, ch_res.single_channel_acc, yerr=ch_res.single_channel_sd, capsize=2, color="#0072BD")
ax[0].axhline(all_acc, c="k", ls="--", lw=0.8, label="All channels"); ax[0].axhline(MAJORITY_RATE, c="r", ls="--", lw=0.8, label="Majority-class rate")
ax[0].set_ylim(0.4, 1.02); ax[0].set_ylabel("Accuracy (mean ± SD across subjects)"); ax[0].set_title("Single-channel classification"); ax[0].legend(fontsize=8)
ax[1].bar(ch_lbl, ch_res.change_when_dropped, color="#D95319"); ax[1].axhline(0, c="k", lw=0.6)
ax[1].set_ylabel("Change in accuracy"); ax[1].set_title("Accuracy change when one channel is removed")
ax[2].errorbar(topk_df.k, topk_df.acc_mean, yerr=topk_df.acc_sd, marker="o", capsize=3); ax[2].axhline(MAJORITY_RATE, c="r", ls="--", lw=0.8)
ax[2].set_xlabel("Number of channels (in-fold top-k)"); ax[2].set_ylabel("Accuracy (mean ± SD across subjects)"); ax[2].set_title("Accuracy versus number of channels")
save_show("channel_contribution_analysis.png")

## 14. Model Feature Importance
Impurity-based importance of a Random Forest (evaluation feature set, Combined by default; baseline-normalized) summed by feature type and by channel. The forest is fitted on all subjects, so these values describe which inputs the fitted model uses; they are exploratory, do not show that a feature generalizes to unseen subjects, and do not establish a physiological cause.

In [ ]:
Xi, yi, cols_i = Xb, yb, cb                       # evaluation dataset from Section 9.2
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=SEED).fit(Xi, yi)
imp = pd.DataFrame({"feature": [c.split("|")[0] for c in cols_i], "channel": [c.split("|")[1] for c in cols_i], "importance": rf.feature_importances_})
by_feat = imp.groupby("feature").importance.sum().sort_values(ascending=False)
by_feat.index = by_feat.index.map(INDEX_LABELS)
by_ch = imp.groupby("channel").importance.sum().reindex(ch_lbl)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
by_feat.plot.bar(ax=ax[0], color="#77AC30"); ax[0].set_title("Importance by feature type (summed over channels)")
ax[0].set_xlabel("Feature"); ax[0].set_ylabel("Summed importance"); plt.setp(ax[0].get_xticklabels(), rotation=30, ha="right")
by_ch.plot.bar(ax=ax[1], color="#7E2F8E"); ax[1].set_title("Importance by channel (summed over features)")
ax[1].set_xlabel("Channel"); ax[1].set_ylabel("Summed importance"); ax[1].tick_params(axis="x", labelrotation=0)
save_show("rf_feature_importance.png")

top_features = imp.assign(feature=imp.feature.map(INDEX_LABELS)).sort_values("importance", ascending=False).head(15)
top_features.to_csv(os.path.join(OUT_DIR, "rf_top_features.csv"), index=False)
display(top_features.head(10).round(4))

## 15. Summary of Classification Performance
Binary early-vs-late classification (LOSO, mean across the 15 held-out subjects) for baseline-normalized and raw features. Accuracy is accompanied by its 95% bootstrap confidence interval over subjects, balanced accuracy, F1 and AUC. The AUC is computed within each held-out subject and averaged; it is insensitive to the decision threshold, whereas accuracy and balanced accuracy depend on a threshold that is shared across subjects.

In [ ]:
def format_summary(d):
    cols = {"feature_set": "Feature set", "model": "Model", "acc_mean": "Accuracy", "acc_sd": "Accuracy SD",
            "acc_ci95_low": "Accuracy 95% CI low", "acc_ci95_high": "Accuracy 95% CI high",
            "bacc_mean": "Balanced accuracy", "f1_mean": "F1", "auc_mean": "AUC"}
    out = d[list(cols)].rename(columns=cols).reset_index(drop=True)
    out["Feature set"] = out["Feature set"].map(lambda k: FS_DISPLAY.get(k, k))
    return out.round(3)

report_norm = format_summary(summ_c[summ_c.baseline_normalized])
report_raw = format_summary(summ_c[~summ_c.baseline_normalized])
print("Binary early-vs-late classification, LOSO, baseline-normalized features:")
display(report_norm)
print("Binary early-vs-late classification, LOSO, raw features:")
display(report_raw)
report_norm.to_csv(os.path.join(OUT_DIR, "classification_summary_normalized.csv"), index=False)
report_raw.to_csv(os.path.join(OUT_DIR, "classification_summary_raw.csv"), index=False)
print("Complete intermediate tables saved to:", OUT_DIR)

## 16. Limitations and Interpretation
- The analysis evaluates how EMG features change over recording time. Early and late labels are defined by time and are proxies for fatigue-related states; no independent measurement of fatigue (force, MVC, time to failure, perceived exertion) is available.
- Classification performance shows that early and late segments are separable from EMG features in held-out subjects. It does not establish clinical fatigue-detection validity, and any monotonic change during the recording (for example electrode or skin changes) could contribute.
- LOSO assesses generalization to a held-out subject under the implemented calibration procedure. Baseline normalization requires an initial baseline segment (the first 10 s) from every subject; without it, absolute feature levels differ between subjects and a threshold shared across subjects transfers less well, which is reflected in the raw-feature results.
- With 15 subjects, subject-level confidence intervals and tests are wide and descriptive. Windows overlap in time and are not independent observations.
- MDF decreased on average, but individual MDF trajectories varied considerably and a straight line described many of them poorly (Section 3.1); the non-linear models (Section 6) are descriptive.
- The fatigue-zone score and the fatigue-response ranking (Sections 5 and 7) are exploratory and have not been validated as measures of fatigue susceptibility. Random Forest importance (Section 14) is global and exploratory.
- Several design choices (window length, label fraction, baseline length) were fixed in advance; Section 12 shows how results vary with window length and label fraction.

## 17. Export of Key Results
Only results that are useful outside the notebook are exported:
- **Figures** (`results/figures/`): the 8 figures listed in `KEY_FIGURES` (Section 1.1), which together show the signal analysis, the limits of the linear model, the spectral-index and fatigue-zone analyses, the trajectory models and the classification results.
- **Tables** (`results/tables/`): one CSV per key table, plus `key_results_summary.csv` / `.md` (the headline numbers in one place) and, if `openpyxl` is installed, `key_tables.xlsx` with every key table on its own sheet.
- Everything else produced above (per-subject and per-channel tables, permutation null distribution, epoch features) stays in `results/supplementary/`.

In [ ]:
# ---- Headline numbers ----
sm_mdf, sm_rms = d_mdf.mean(1), d_rms.mean(1)                       # one mean slope per subject
def mean_ci_p(v):
    lo, hi = stats.t.interval(0.95, len(v) - 1, loc=v.mean(), scale=stats.sem(v))
    return f"{v.mean():.4f} (95% CI {lo:.4f} to {hi:.4f}), p = {stats.ttest_1samp(v, 0).pvalue:.2e}"

norm_cfg = summ_c[summ_c.baseline_normalized & (summ_c.feature_set != BASELINE_SET)]
raw_cfg = summ_c[~summ_c.baseline_normalized & (summ_c.feature_set != BASELINE_SET)]
top_n, top_r = norm_cfg.loc[norm_cfg.bacc_mean.idxmax()], raw_cfg.loc[raw_cfg.bacc_mean.idxmax()]
pre = summ_c[summ_c.baseline_normalized & (summ_c.feature_set == EVAL_FEATURE_SET) & (summ_c.model == EVAL_MODEL)].iloc[0]
ref = summ_c[summ_c.baseline_normalized & (summ_c.feature_set == BASELINE_SET)].iloc[0]

rows = [
    ("Dataset", "Subjects / epochs per subject / channels", f"{S} / {E} (1 s) / {N_CH}"),
    ("Signal analysis", "Mean MDF, first 10 s -> last 10 s (Hz)", f"{mdf_ep[:, :10].mean():.1f} -> {mdf_ep[:, -10:].mean():.1f}"),
    ("Signal analysis", "Overall ΔMDF, mean of subject means (Hz/s)", mean_ci_p(sm_mdf)),
    ("Signal analysis", "Overall ΔRMS, mean of subject means (RMS units/s)", mean_ci_p(sm_rms)),
    ("Signal analysis", "Series with ΔRMS > 0 / ΔMDF < 0 (%)", f"{100 * np.mean(d_rms > 0):.1f} / {100 * np.mean(d_mdf < 0):.1f}"),
    ("Signal analysis", "Median linear-fit R²: RMS / MDF", f"{np.median(r2_rms):.3f} / {np.median(r2_mdf):.3f}"),
    ("Signal analysis", "Series with R² < 0.1 (%): RMS / MDF", f"{100 * np.mean(r2_rms < 0.1):.1f} / {100 * np.mean(r2_mdf < 0.1):.1f}"),
    ("Signal analysis", "Channels significant after Holm (of 8): ΔMDF / ΔRMS", f"{int(tab['mdf_channel'].sig_holm.sum())} / {int(tab['rms_channel'].sig_holm.sum())}"),
    ("Signal analysis", "Subjects significant after Holm (of 15, descriptive): ΔMDF / ΔRMS", f"{int(tab['mdf_subject'].sig_holm.sum())} / {int(tab['rms_subject'].sig_holm.sum())}"),
    ("Signal analysis", "MDF-based zone share of series in the Fatigue zone (%)", f"{qshare.loc['MDF', 'Fatigue']:.1f}"),
    ("Signal analysis", "Series where a piecewise model beats linear (%): MDF / RMS", f"{summ.loc['MDF', 'pct_piecewise_beats_linear']:.1f} / {summ.loc['RMS', 'pct_piecewise_beats_linear']:.1f}"),
    ("Signal analysis", "Series with a valid onset (%): MDF / RMS", f"{summ.loc['MDF', 'pct_valid_onset']:.1f} / {summ.loc['RMS', 'pct_valid_onset']:.1f}"),
    ("Classification (binary, LOSO)", "Windows / majority-class accuracy", f"{len(yb)} / {MAJORITY_RATE:.3f}"),
    ("Classification (binary, LOSO)", "Highest balanced accuracy, baseline-normalized (selected after seeing results)",
     f"{top_n.feature_set} + {top_n.model}: balanced acc. {top_n.bacc_mean:.3f}, accuracy {top_n.acc_mean:.3f} (95% CI {top_n.acc_ci95_low:.3f} to {top_n.acc_ci95_high:.3f}), AUC {top_n.auc_mean:.3f}"),
    ("Classification (binary, LOSO)", "Pre-specified configuration (permutation-tested)",
     f"{EVAL_FEATURE_SET} + {EVAL_MODEL}, baseline-normalized: balanced acc. {obs:.3f}; null mean {null.mean():.3f}, null 95th pct {np.percentile(null, 95):.3f}; permutation p = {p_perm:.4f}; sign-flip p = {p_flip:.5f}"),
    ("Classification (binary, LOSO)", "Highest balanced accuracy, raw features",
     f"{top_r.feature_set} + {top_r.model}: balanced acc. {top_r.bacc_mean:.3f}, accuracy {top_r.acc_mean:.3f}"),
    ("Classification (binary, LOSO)", "MDF-only reference, baseline-normalized", f"balanced acc. {ref.bacc_mean:.3f}, accuracy {ref.acc_mean:.3f}"),
]
for _, r_ in pc[pc.comparison.str.contains("normalized vs raw")].iterrows():
    rows.append(("Classification (binary, LOSO)", f"Normalization effect on accuracy, {r_.comparison}", f"{r_.mean_diff:+.3f} (Holm p = {r_.p_holm:.4f})"))
rows.append(("Classification (three-class, LOSO)", "Highest macro F1 configuration",
             f"{b3.feature_set} + {b3.model}: balanced acc. {b3.bacc_mean:.3f}, macro F1 {b3.macro_f1:.3f}, accuracy {b3.acc_mean:.3f}"))
key_summary = pd.DataFrame(rows, columns=["Section", "Metric", "Value"])
display(key_summary)

# ---- Key tables ----
slope_long = pd.concat([df.reset_index().rename(columns={"label": "unit"}).assign(signal=k.split("_")[0].upper(), level=k.split("_")[1])
                        for k, df in tab.items()], ignore_index=True)
slope_long["slope_unit"] = np.where(slope_long.signal == "MDF", "Hz/s", "RMS units/s")
slope_long = slope_long[["signal", "level", "unit", "slope_unit"] + [c for c in slope_long.columns if c not in ("signal", "level", "unit", "slope_unit")]]

key_tables = {
    "key_results_summary": key_summary,
    "qc_report": qc,
    "slope_statistics": slope_long,
    "spectral_index_comparison": idx_cmp.rename_axis("spectral_index").reset_index(),
    "fatigue_zone_subject_scores": subj.rename_axis("subject").reset_index(),
    "fatigue_zone_channel_fractions": chan.rename_axis("channel").reset_index(),
    "trajectory_model_summary": summ.reset_index(),
    "fatigue_response_ranking": ranking.rename_axis("subject").reset_index(),
    "binary_classification_normalized": report_norm,
    "binary_classification_raw": report_raw,
    "binary_paired_comparisons": pc,
    "three_class_summary": res3,
    "sensitivity_analysis": sens,
    "channel_contribution": ch_res.reset_index(),
}
for name, df in key_tables.items():
    df.to_csv(os.path.join(TABLE_DIR, f"{name}.csv"), index=False, float_format="%.6g")

# Headline summary as a Markdown table (for the README)
md_lines = ["| Section | Metric | Value |", "|---|---|---|"] + [f"| {a} | {b} | {c} |" for a, b, c in rows]
with open(os.path.join(TABLE_DIR, "key_results_summary.md"), "w", encoding="utf-8") as fh:
    fh.write("\n".join(md_lines) + "\n")

# Optional Excel workbook with one sheet per key table
try:
    with pd.ExcelWriter(os.path.join(TABLE_DIR, "key_tables.xlsx")) as xw:
        for name, df in key_tables.items():
            df.to_excel(xw, sheet_name=name[:31], index=False)
except ImportError:
    print("openpyxl not installed: key_tables.xlsx was skipped (CSV files were written).")

missing = [f for f in KEY_FIGURES if not os.path.exists(os.path.join(FIG_DIR, f))]
assert not missing, f"Key figures missing from {FIG_DIR}: {missing}"
print("Figures in", FIG_DIR, ":", sorted(os.listdir(FIG_DIR)))
print("Tables in", TABLE_DIR, ":", sorted(os.listdir(TABLE_DIR)))